# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [1]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [7]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    x = x.detach().requires_grad_(True)
    y = torch.sum(x ** 2)
    return torch.autograd.grad(y, x)[0]

x = torch.tensor([[3.0, 6.0], [2.0, 3.0]], requires_grad=True)
grad = grad_with_autograd_grad(x)
grad

tensor([[ 6., 12.],
        [ 4.,  6.]])

In [4]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    x = x.detach().requires_grad_(True)
    y = torch.sum(x ** 2)
    y.backward()
    return x.grad

x = torch.tensor([[3.0, 6.0], [2.0, 3.0]], requires_grad=True)
grad = grad_with_backward(x)
grad

tensor([[ 6., 12.],
        [ 4.,  6.]])

In [10]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    a = a.detach().requires_grad_(True)
    b = b.detach().requires_grad_(True)
    y = torch.sum(a ** 2 + a * b)
    grad_a, grad_b = torch.autograd.grad(y, (a, b))
    return grad_a, grad_b

a = torch.tensor([4.0, 9.0], requires_grad=True)
b = torch.tensor([3.0, 2.0], requires_grad=True)
grad_a, grad_b = grad_wrt_multiple_inputs(a, b)
grad_a, grad_b

(tensor([11., 20.]), tensor([4., 9.]))

## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [11]:
from torch.utils.data import DataLoader, Dataset

In [16]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.x[idx], self.y[idx]
    
    
x = torch.arange(10).float().view(-1, 1)
y = 2 * x + 1
dataset = TensorPairDataset(x, y)
dataset[0], dataset[1], len(dataset)

((tensor([0.]), tensor([1.])), (tensor([1.]), tensor([3.])), 10)

In [18]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        tokens = tokens.detach().clone()
        self.input_ids = tokens[:, :-1]
        self.target_ids = tokens[:, 1:]

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.input_ids[idx], self.target_ids[idx]
      
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
dataset = NextTokenDataset(x)
dataset[0], dataset[1], len(dataset)
dataset.input_ids ,dataset.target_ids


(tensor([[1, 2],
         [4, 5]]),
 tensor([[2, 3],
         [5, 6]]))

In [23]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        self.tokens = tokens
        self.crop_len = crop_len
        self.generator = torch.Generator()
        if seed is not None:
            self.generator.manual_seed(seed)

    def __len__(self) -> int:
        return len(self.tokens)

    def __getitem__(self, idx: int) -> torch.Tensor:
        T_total = self.tokens.size(1)
        s = torch.randint(0, T_total - self.crop_len + 1, (), generator=self.generator).item()
        return self.tokens[idx, s:s+self.crop_len]
    

x = torch.arange(30).view(6, 5)
dataset = RandomCropSequenceDataset(x, crop_len=4, seed=42)
dataset[0], dataset[1], len(dataset)


(tensor([0, 1, 2, 3]), tensor([6, 7, 8, 9]), 6)

In [24]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    lengths = torch.tensor([seq.size(0) for seq in seqs], dtype=torch.long)
    t_max = lengths.max().item()
    tokens = torch.full((len(seqs), t_max), pad_value, dtype=torch.long)
    padding_mask = torch.zeros((len(seqs), t_max), dtype=torch.bool)
    for i, seq in enumerate(seqs):
        padding_mask[i, lengths[i]:] = True
        tokens[i, :lengths[i]] = seq.long()
    return PaddedBatch(tokens=tokens, lengths=lengths, padding_mask=padding_mask)

seqs = [torch.tensor([1, 2, 3]), torch.tensor([4, 5]), torch.tensor([6])]
padded_batch = pad_1d_sequences(seqs, pad_value=0)
padded_batch.tokens, padded_batch.lengths, padded_batch.padding_mask

(tensor([[1, 2, 3],
         [4, 5, 0],
         [6, 0, 0]]),
 tensor([3, 2, 1]),
 tensor([[False, False, False],
         [False, False,  True],
         [False,  True,  True]]))

In [25]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    input_ids_seqs, target_ids_seqs = zip(*batch)
    padded_input = pad_1d_sequences(input_ids_seqs, pad_value=pad_value)
    padded_target = pad_1d_sequences(target_ids_seqs, pad_value=pad_value)

    attention_mask = ~padded_input.padding_mask

    return {
        "input_ids": padded_input.tokens,
        "target_ids": padded_target.tokens,
        "attention_mask": attention_mask,
        "padding_mask": padded_input.padding_mask,
    }
    
    
batch = [(torch.tensor([1, 2, 3]), torch.tensor([2, 3, 4])),
         (torch.tensor([5, 6]), torch.tensor([6, 7])),
         (torch.tensor([8]), torch.tensor([9]))]
collated = collate_next_token_batch(batch, pad_value=0)
collated["input_ids"], collated["target_ids"], collated["attention_mask"], collated["padding_mask"]

(tensor([[1, 2, 3],
         [5, 6, 0],
         [8, 0, 0]]),
 tensor([[2, 3, 4],
         [6, 7, 0],
         [9, 0, 0]]),
 tensor([[ True,  True,  True],
         [ True,  True, False],
         [ True, False, False]]),
 tensor([[False, False, False],
         [False, False,  True],
         [False,  True,  True]]))

In [31]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,
        num_workers=num_workers,
    )

## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [27]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    with torch.no_grad():
        m = torch.zeros_like(p)
        v = torch.zeros_like(p)
    return AdamWState(m=m, v=v, t=0)
  
adams = init_adamw_state(torch.tensor([1.0, 2.0]))


In [28]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    state.t += 1
    beta1, beta2 = betas
    state.m = beta1 * state.m + (1 - beta1) * grad
    state.v = beta2 * state.v + (1 - beta2) * grad * grad
    m_hat = state.m / (1 - beta1 ** state.t)
    v_hat = state.v / (1 - beta2 ** state.t)
    p.data = p.data - lr * (m_hat / (torch.sqrt(v_hat) + eps) + weight_decay * p.data)
    return state
  
adams = init_adamw_state(torch.tensor([1.0, 2.0]))
adamw_step_(torch.tensor([1.0, 2.0]), torch.tensor([0.1, 0.2]), adams, lr=0.01)

AdamWState(m=tensor([0.0100, 0.0200]), v=tensor([1.0000e-05, 4.0000e-05]), t=1)

In [32]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    if not (len(params) == len(grads) == len(states)):
        raise ValueError("params, grads, and states must have the same length")
    for i in range(len(params)):
        states[i] = adamw_step_(params[i], grads[i], states[i], lr, betas, eps, weight_decay)
    return states

adams = init_adamw_state(torch.tensor([1.0, 2.0]))
adamw_step_many_([torch.tensor([1.0, 2.0]), torch.tensor([3.0, 4.0])],
                  [torch.tensor([0.1, 0.2]), torch.tensor([0.3, 0.4])],
                  [adams, adams],
                  lr=0.01)


[AdamWState(m=tensor([0.0390, 0.0580]), v=tensor([9.9990e-05, 1.9996e-04]), t=2),
 AdamWState(m=tensor([0.0390, 0.0580]), v=tensor([9.9990e-05, 1.9996e-04]), t=2)]

## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [ ]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    model.train()
    optimizer.zero_grad()
    inputs, targets = batch
    outputs = model(inputs)
    loss = nn.functional.mse_loss(outputs, targets)
    loss.backward()
    optimizer.step()
    return loss.item()



Loss: 460.7028
Loss: 454.8325
Loss: 449.0409
Loss: 443.3267
Loss: 437.6890
Loss: 432.1268
Loss: 426.6390
Loss: 421.2247
Loss: 415.8828
Loss: 410.6124
Loss: 405.4126
Loss: 400.2824
Loss: 395.2208
Loss: 390.2270
Loss: 385.3000
Loss: 380.4389
Loss: 375.6430
Loss: 370.9112
Loss: 366.2427
Loss: 361.6367
Loss: 357.0923
Loss: 352.6088
Loss: 348.1852
Loss: 343.8209
Loss: 339.5150
Loss: 335.2668
Loss: 331.0753
Loss: 326.9399
Loss: 322.8599
Loss: 318.8344
Loss: 314.8629
Loss: 310.9445
Loss: 307.0786
Loss: 303.2643
Loss: 299.5012
Loss: 295.7883
Loss: 292.1252
Loss: 288.5110
Loss: 284.9453
Loss: 281.4273
Loss: 277.9563
Loss: 274.5318
Loss: 271.1531
Loss: 267.8196
Loss: 264.5307
Loss: 261.2859
Loss: 258.0844
Loss: 254.9257
Loss: 251.8094
Loss: 248.7347
Loss: 245.7012
Loss: 242.7083
Loss: 239.7554
Loss: 236.8420
Loss: 233.9676
Loss: 231.1316
Loss: 228.3337
Loss: 225.5731
Loss: 222.8495
Loss: 220.1622
Loss: 217.5110
Loss: 214.8952
Loss: 212.3145
Loss: 209.7682
Loss: 207.2560
Loss: 204.7775
Loss: 202.

C:\Users\USR1\AppData\Local\Temp\ipykernel_29332\3167300395.py:13: UserWarning: Using a target size (torch.Size([100, 1])) that is different to the input size (torch.Size([100, 10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = nn.functional.mse_loss(outputs, targets)


RuntimeError: a Tensor with 10 elements cannot be converted to Scalar

## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [60]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    return weight.size(1), weight.size(0)  # (fan_in, fan_out)

In [61]:


def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * (6 / (fan_in + fan_out))**0.5
    return torch.empty_like(weight).uniform_(-bound, bound)

In [62]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    if nonlinearity == "relu":
        gain = 2**0.5
    else:
        raise ValueError(f"Unsupported nonlinearity: {nonlinearity}")
    fan_in, _ = fan_in_fan_out(weight)
    std = gain / (fan_in**0.5)
    bound = (3**0.5) * std
    return torch.empty_like(weight).uniform_(-bound, bound)

In [80]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    if scheme == "xavier":
        xavier_uniform_(layer.weight)
    elif scheme == "kaiming_relu":
        kaiming_uniform_(layer.weight, nonlinearity="relu")
    elif scheme == "zero":
        with torch.no_grad():
            layer.weight.zero_()
            if layer.bias is not None:
                layer.bias.zero_()
    else:
        raise ValueError(f"Unknown scheme: {scheme}")
    return layer
  
  

